# Full SEN2SR: Pretrained Baseline vs. Fine-Tuned 10-Band Comparison
### 3-Way Comparative Evaluation: Native 10m vs. Baseline Full (2.5m) vs. Fine-Tuned Full (2.5m)

This notebook executes an end-to-end analytical comparison on a real 10-band Sentinel-2 Level-2A acquisition:
$$\text{Native Sentinel-2 (10m)} \quad \longleftrightarrow \quad \text{Baseline Full SEN2SR (2.5m)} \quad \longleftrightarrow \quad \text{Fine-Tuned Full SEN2SR (2.5m)}$$

---

### Core Innovations & Problem Solutions in this Notebook
1. **Bulletproof Mamba Architecture:** Full SEN2SR Stage 1 uses `MambaSR` (Vision State Space Model). This notebook incorporates a self-contained fallback ensuring that if Colab exhibits C++ ABI/CUDA symbol mismatches with `selective_scan_cuda`, the pure-PyTorch selective scan executes seamlessly without crashing.
2. **Fixed "Last Tile" Boundary Engine:** Previous implementations cut off right/bottom boundary tiles when scene dimensions were not exact multiples of the tile stride. This engine guarantees complete coverage of the final edges (`x_steps` & `y_steps` strictly reach boundaries) and smoothly blends all tiles using a **2D Hann spatial cosine window** for seam-free reconstruction.
3. **Complete 10-Band Stack:** Super-resolves all 10 analytical bands (`B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`) across both models using Swin Transformer V2 guidance.
4. **All 5 Scientific Index Layers:** Evaluates True Color RGB, False Color Infrared, NDVI (Vegetation), NDWI (Water/Moisture), and NDBI (Built-up Urban Index).


## Cell 1 — Mount Google Drive & Environment Setup


In [ ]:
import os, sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CPL_LOG_ERRORS'] = 'OFF'
os.environ['CPL_LOG'] = 'OFF'
os.environ['GDAL_PAM_ENABLED'] = 'NO'

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import glob, re, zipfile, shutil, time, warnings, logging
warnings.filterwarnings('ignore')
for logger_name in ['rasterio', 'rasterio._env', 'rasterio.env', 'rasterio._base']:
    l = logging.getLogger(logger_name)
    l.setLevel(logging.ERROR)
    l.propagate = False

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from affine import Affine
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.cuda.empty_cache()

BASE_DIR = '/content/drive/MyDrive/Project_Folder'
DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
RESULTS_DIR = os.path.join(BASE_DIR, 'Results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Select dataset ZIP
zip_candidates = [
    os.path.join(DATASETS_DIR, 'Browser_images_1.zip'),
    os.path.join(DATASETS_DIR, 'Browser_images.zip'),
    os.path.join(DATASETS_DIR, 'Browser_images_2.zip')
]
ZIP_PATH = None
for z in zip_candidates:
    if os.path.exists(z):
        ZIP_PATH = z
        break

assert ZIP_PATH is not None, f"No Copernicus ZIP found in {DATASETS_DIR}! Please add Browser_images_1.zip."
print(f'Using Copernicus archive: {ZIP_PATH}')

CHECKPOINT_PATH = os.path.join(RESULTS_DIR, 'sen2sr_full_mamba_compound_finetuned.pth')
print(f'Target fine-tuned checkpoint: {CHECKPOINT_PATH}')
print(f'Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}')


## Cell 2 — Install Required Packages & Initialize Mamba Engine


In [ ]:
%pip install -q sen2sr mlstac rasterio safetensors einops timm

import types, sys
from einops import rearrange, repeat

# -----------------------------------------------------------------------------------------
# Robust Mamba-SSM Import Fallback (prevents Colab undefined symbol crash)
# -----------------------------------------------------------------------------------------
mamba_ready = False

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    test_u = torch.randn(1, 16, 32, device='cuda' if torch.cuda.is_available() else 'cpu')
    test_delta = torch.randn(1, 16, 32, device=test_u.device)
    test_A = -torch.rand(16, 16, device=test_u.device)
    test_B = torch.randn(1, 16, 32, device=test_u.device)
    test_C = torch.randn(1, 16, 32, device=test_u.device)
    _ = selective_scan_fn(test_u, test_delta, test_A, test_B, test_C,
                          delta_bias=torch.zeros(16, device=test_u.device),
                          delta_softplus=True)
    mamba_ready = True
    print('Native CUDA Mamba-SSM kernel loaded and verified successfully!')
except Exception as e:
    print(f'Note: Native CUDA selective_scan_cuda could not be loaded ({type(e).__name__}).')
    print('Activating self-contained pure-PyTorch selective scan implementation...')

if not mamba_ready:
    def selective_scan_ref(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False, return_last_state=False):
        dtype_in = u.dtype
        u = u.float()
        delta = delta.float()
        if delta_bias is not None:
            delta = delta + delta_bias[..., None].float()
        if delta_softplus:
            delta = F.softplus(delta)
        batch, dim, dstate = u.shape[0], A.shape[0], A.shape[1]
        is_variable_B = B.dim() >= 3
        is_variable_C = C.dim() >= 3
        if A.is_complex():
            if is_variable_B:
                B = torch.view_as_complex(rearrange(B.float(), "... (L two) -> ... L two", two=2))
            if is_variable_C:
                C = torch.view_as_complex(rearrange(C.float(), "... (L two) -> ... L two", two=2))
        else:
            B = B.float()
            C = C.float()
        x = A.new_zeros((batch, dim, dstate))
        ys = []
        deltaA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A))
        if not is_variable_B:
            deltaB_u = torch.einsum('bdl,dn,bdl->bdln', delta, B, u)
        else:
            if B.dim() == 3:
                deltaB_u = torch.einsum('bdl,bnl,bdl->bdln', delta, B, u)
            else:
                B = repeat(B, "B G N L -> B (G H) N L", H=dim // B.shape[1])
                deltaB_u = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)
        if is_variable_C and C.dim() == 4:
            C = repeat(C, "B G N L -> B (G H) N L", H=dim // C.shape[1])
        last_state = None
        for i in range(u.shape[2]):
            x = deltaA[:, :, i] * x + deltaB_u[:, :, i]
            if not is_variable_C:
                y = torch.einsum('bdn,dn->bd', x, C)
            else:
                if C.dim() == 3:
                    y = torch.einsum('bdn,bn->bd', x, C[:, :, i])
                else:
                    y = torch.einsum('bdn,bdn->bd', x, C[:, :, :, i])
            if i == u.shape[2] - 1:
                last_state = x
            if y.is_complex():
                y = y.real * 2
            ys.append(y)
        y = torch.stack(ys, dim=2)
        out = y if D is None else y + u * rearrange(D, "d -> d 1")
        if z is not None:
            out = out * F.silu(z)
        out = out.to(dtype=dtype_in)
        return out if not return_last_state else (out, last_state)

    mod_ops = types.ModuleType('mamba_ssm.ops.selective_scan_interface')
    mod_ops.selective_scan_fn = selective_scan_ref
    mod_ops.selective_scan_ref = selective_scan_ref
    sys.modules['mamba_ssm.ops.selective_scan_interface'] = mod_ops

    mod_mamba = types.ModuleType('mamba_ssm')
    mod_mamba.ops = types.ModuleType('mamba_ssm.ops')
    mod_mamba.ops.selective_scan_interface = mod_ops
    sys.modules['mamba_ssm'] = mod_mamba
    sys.modules['mamba_ssm.ops'] = mod_mamba.ops
    print('Pure-PyTorch selective scan fallback registered successfully!')

from sen2sr.models.opensr_baseline.mamba import MambaSR
print('MambaSR successfully loaded and ready for full comparison pipeline!')


## Cell 3 — Extract 10-Band ZIP Archive


In [ ]:
EXTRACT_DIR = '/content/copernicus_data'
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

print(f'Archive extracted to: {EXTRACT_DIR}')
raster_files = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith(('.tif', '.tiff', '.jp2')):
            raster_files.append(os.path.join(root, f))

print(f'Found {len(raster_files)} raster files.')


## Cell 4 — Identify & Stack All 10 Sentinel-2 Bands


In [ ]:
BAND_NAMES = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
found = {}

for path in raster_files:
    fname = os.path.basename(path).upper()
    for b in BAND_NAMES:
        if re.search(rf'(^|[_\.-]){b}([_\.-]|$)', fname):
            found[b] = path
            break

missing = [b for b in BAND_NAMES if b not in found]
assert not missing, f"Missing required bands: {missing}! Ensure all 10 bands are present in archive."

print('All 10 Sentinel-2 bands matched successfully:')
for b in BAND_NAMES:
    print(f'  {b:4s} -> {os.path.basename(found[b])}')

# Reference grid: B02 (10m)
with rasterio.open(found['B02']) as ref:
    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    height, width = ref.height, ref.width

stack = []
for b in BAND_NAMES:
    with rasterio.open(found[b]) as src:
        dest = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )
        stack.append(dest)

raw_stack = np.stack(stack).astype(np.float32)
finite = raw_stack[np.isfinite(raw_stack)]
p99 = np.percentile(finite, 99)
SCALE_FACTOR = 10000.0 if p99 > 100.0 else 1.0

X_np = np.clip(raw_stack / SCALE_FACTOR, 0.0, 1.0)
X = torch.from_numpy(X_np).float().to(device)

print(f'\nStacked 10-band tensor shape: {tuple(X.shape)} (Channels, H, W)')
print(f'Pixel reflectance range: [{X_np.min():.4f}, {X_np.max():.4f}]')


## Cell 5 — Load Both Pretrained Baseline & Fine-Tuned 10-Band Full Generators


In [ ]:
import mlstac
import safetensors.torch

MODEL_DIR = '/content/model/SEN2SR_FULL'
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Download official full model weights if not present
mlstac.download(
    file='https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SR/main/mlm.json',
    output_dir=MODEL_DIR
)

# 2. Load Pretrained Baseline 10-band compiled model
print('Loading Baseline 10-Band SEN2SR Full model...')
model_baseline = mlstac.load(MODEL_DIR).compiled_model(device=device)
model_baseline = model_baseline.to(device).eval()
print('Pretrained Baseline 10-Band SEN2SR Full loaded successfully.')

# 3. Load Fine-Tuned model and replace Stage 1 MambaSR weights
print('\nLoading Fine-Tuned 10-Band SEN2SR Full model...')
model_finetuned = mlstac.load(MODEL_DIR).compiled_model(device=device)

if os.path.exists(CHECKPOINT_PATH):
    weights_finetuned = torch.load(CHECKPOINT_PATH, map_location=device)
    injected = False

    # SEN2SR Full hierarchy: model_finetuned (rswir_modelx4) -> sr_model (rgbn_model) -> sr_model (MambaSR)
    if hasattr(model_finetuned, 'sr_model') and hasattr(model_finetuned.sr_model, 'sr_model'):
        try:
            model_finetuned.sr_model.sr_model.load_state_dict(weights_finetuned)
            injected = True
            print('   - Injected fine-tuned weights directly into model.sr_model.sr_model (MambaSR)!')
        except Exception as e:
            print(f'   - Direct hierarchy load failed: {e}. Searching submodules...')

    if not injected:
        for name, m in model_finetuned.named_modules():
            if type(m).__name__ == 'MambaSR':
                try:
                    m.load_state_dict(weights_finetuned)
                    injected = True
                    print(f"   - Injected fine-tuned weights into '{name}' (MambaSR)!")
                    break
                except Exception:
                    continue

    assert injected, "CRITICAL ERROR: Failed to inject fine-tuned weights into SEN2SR Full Stage 1 MambaSR!"

    # Verify mathematical divergence between fine-tuned and baseline weights
    p_base = next(model_baseline.sr_model.sr_model.conv_after_body.parameters())
    p_fine = next(model_finetuned.sr_model.sr_model.conv_after_body.parameters())
    weight_diff_norm = torch.norm(p_fine - p_base).item()
    print(f'Weight Difference Norm (Fine-Tuned vs Baseline): {weight_diff_norm:.6f}')
    assert weight_diff_norm > 1e-4, f'CRITICAL: Weights did not update (diff={weight_diff_norm})! Models are identical.'
else:
    print(f'Warning: Checkpoint {CHECKPOINT_PATH} not found. Running comparative demonstration with baseline weights.')

model_finetuned = model_finetuned.to(device).eval()
print('Both 10-Band Full Models initialized and ready!')


## Cell 6 — Full 10-Band Inference with 2D Hann Window Blending


In [ ]:
def run_tiled_inference_hann(input_tensor, model, patch_size=96, overlap=24):
    """
    Performs seamless tiled inference with 2D Hann window cosine blending.
    Guarantees that boundary / 'last tile' edges are strictly reached and artifacts eliminated.
    Uses AMP FP16 and memory-efficient tile execution.
    """
    _, H, W = input_tensor.shape
    stride = patch_size - overlap
    scale = 4  # 10m -> 2.5m

    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)
    if pad_h > 0 or pad_w > 0:
        x_pad = F.pad(input_tensor, (0, pad_w, 0, pad_h), mode='replicate')
    else:
        x_pad = input_tensor

    _, H_pad, W_pad = x_pad.shape

    y_steps = list(range(0, H_pad - patch_size + 1, stride))
    if not y_steps or y_steps[-1] != H_pad - patch_size:
        y_steps.append(H_pad - patch_size)

    x_steps = list(range(0, W_pad - patch_size + 1, stride))
    if not x_steps or x_steps[-1] != W_pad - patch_size:
        x_steps.append(W_pad - patch_size)

    out_bands = 10
    out_accum = torch.zeros((out_bands, H_pad * scale, W_pad * scale), device=device, dtype=torch.float32)
    weight_accum = torch.zeros((1, H_pad * scale, W_pad * scale), device=device, dtype=torch.float32)

    hann_1d_y = torch.hann_window(patch_size * scale, periodic=False, device=device, dtype=torch.float32)
    hann_1d_x = torch.hann_window(patch_size * scale, periodic=False, device=device, dtype=torch.float32)
    window_2d = (hann_1d_y.unsqueeze(1) * hann_1d_x.unsqueeze(0)).unsqueeze(0).clamp_min(1e-3)

    print(f'Executing tiled inference across {len(y_steps)} x {len(x_steps)} = {len(y_steps)*len(x_steps)} tiles...')

    with torch.inference_mode():
        for y0 in y_steps:
            for x0 in x_steps:
                patch = x_pad[:, y0:y0+patch_size, x0:x0+patch_size].unsqueeze(0)
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    pred = model(patch).squeeze(0).float()

                oy, ox = y0 * scale, x0 * scale
                ph, pw = pred.shape[-2:]

                out_accum[:, oy:oy+ph, ox:ox+pw] += pred * window_2d
                weight_accum[:, oy:oy+ph, ox:ox+pw] += window_2d

    super_res = out_accum / weight_accum.clamp_min(1e-6)
    super_res = super_res[:, :H * scale, :W * scale]
    return super_res

print('Running Pretrained Full Baseline Inference...')
t0 = time.time()
output_baseline = run_tiled_inference_hann(X, model_baseline)
print(f'Baseline Inference completed in {time.time() - t0:.2f}s | Output: {tuple(output_baseline.shape)}')

if device.type == 'cuda':
    torch.cuda.empty_cache()

print('\nRunning Fine-Tuned Full Inference...')
t0 = time.time()
output_finetuned = run_tiled_inference_hann(X, model_finetuned)
print(f'Fine-Tuned Inference completed in {time.time() - t0:.2f}s | Output: {tuple(output_finetuned.shape)}')


## Cell 7 — Compute All 5 Scientific Layers (RGB, CIR, NDVI, NDWI, NDBI)


In [ ]:
BAND_MAP = {name: i for i, name in enumerate(BAND_NAMES)}

def calc_indices_torch(tensor):
    def norm_diff(a, b, eps=1e-6):
        denom = a + b
        valid = (torch.abs(denom) > eps) & torch.isfinite(a) & torch.isfinite(b)
        res = torch.full_like(denom, float('nan'))
        res[valid] = (a[valid] - b[valid]) / denom[valid]
        return res

    b02 = tensor[BAND_MAP['B02']]
    b03 = tensor[BAND_MAP['B03']]
    b04 = tensor[BAND_MAP['B04']]
    b08 = tensor[BAND_MAP['B08']]
    b11 = tensor[BAND_MAP['B11']]

    # RGB True Color
    rgb = torch.stack([b04, b03, b02], dim=0)
    # False Color Infrared (CIR)
    cir = torch.stack([b08, b04, b03], dim=0)
    # Indices
    ndvi = norm_diff(b08, b04)
    ndwi = norm_diff(b03, b08)
    ndbi = norm_diff(b11, b08)

    return {'rgb': rgb, 'cir': cir, 'ndvi': ndvi, 'ndwi': ndwi, 'ndbi': ndbi}

layers_native = calc_indices_torch(X)
layers_baseline = calc_indices_torch(output_baseline)
layers_finetuned = calc_indices_torch(output_finetuned)

print('All 5 evaluation layers computed successfully across all 3 models!')


## Cell 8 — Side-by-Side 3-Way Visual Comparison


In [ ]:
def to_display_rgb(t):
    """Robust 2% - 98% percentile min-max stretching for natural contrast without white clipping."""
    arr = t.detach().cpu().numpy()
    if arr.ndim == 3 and arr.shape[0] in [3, 4]:
        arr = arr[:3].transpose(1, 2, 0).astype(np.float32)
    else:
        arr = arr.astype(np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(finite, [2, 98])
    if hi > lo:
        return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return np.clip(arr, 0.0, 1.0)

fig, axes = plt.subplots(5, 3, figsize=(18, 25))

titles_col = [
    'Native Sentinel-2 (10m Input)',
    'Pretrained Full Baseline (2.5m)',
    'Fine-Tuned Full Model (2.5m)'
]

# Row 1: True Color RGB
axes[0, 0].imshow(to_display_rgb(layers_native['rgb']))
axes[0, 1].imshow(to_display_rgb(layers_baseline['rgb']))
axes[0, 2].imshow(to_display_rgb(layers_finetuned['rgb']))
axes[0, 0].set_ylabel('True Color RGB\n(B04, B03, B02)', fontsize=12, fontweight='bold')

# Row 2: False Color Infrared
axes[1, 0].imshow(to_display_rgb(layers_native['cir']))
axes[1, 1].imshow(to_display_rgb(layers_baseline['cir']))
axes[1, 2].imshow(to_display_rgb(layers_finetuned['cir']))
axes[1, 0].set_ylabel('Color Infrared (CIR)\n(B08, B04, B03)', fontsize=12, fontweight='bold')

# Row 3: NDVI
im_ndvi_0 = axes[2, 0].imshow(layers_native['ndvi'].cpu().numpy(), cmap='RdYlGn', vmin=-0.2, vmax=0.8)
im_ndvi_1 = axes[2, 1].imshow(layers_baseline['ndvi'].cpu().numpy(), cmap='RdYlGn', vmin=-0.2, vmax=0.8)
im_ndvi_2 = axes[2, 2].imshow(layers_finetuned['ndvi'].cpu().numpy(), cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[2, 0].set_ylabel('NDVI\n(Vegetation)', fontsize=12, fontweight='bold')
plt.colorbar(im_ndvi_2, ax=axes[2, 2], fraction=0.046, pad=0.04)

# Row 4: NDWI
im_ndwi_0 = axes[3, 0].imshow(layers_native['ndwi'].cpu().numpy(), cmap='Blues', vmin=-0.5, vmax=0.5)
im_ndwi_1 = axes[3, 1].imshow(layers_baseline['ndwi'].cpu().numpy(), cmap='Blues', vmin=-0.5, vmax=0.5)
im_ndwi_2 = axes[3, 2].imshow(layers_finetuned['ndwi'].cpu().numpy(), cmap='Blues', vmin=-0.5, vmax=0.5)
axes[3, 0].set_ylabel('NDWI\n(Water / Moisture)', fontsize=12, fontweight='bold')
plt.colorbar(im_ndwi_2, ax=axes[3, 2], fraction=0.046, pad=0.04)

# Row 5: NDBI
im_ndbi_0 = axes[4, 0].imshow(layers_native['ndbi'].cpu().numpy(), cmap='PuOr', vmin=-0.5, vmax=0.5)
im_ndbi_1 = axes[4, 1].imshow(layers_baseline['ndbi'].cpu().numpy(), cmap='PuOr', vmin=-0.5, vmax=0.5)
im_ndbi_2 = axes[4, 2].imshow(layers_finetuned['ndbi'].cpu().numpy(), cmap='PuOr', vmin=-0.5, vmax=0.5)
axes[4, 0].set_ylabel('NDBI\n(Built-up Impervious)', fontsize=12, fontweight='bold')
plt.colorbar(im_ndbi_2, ax=axes[4, 2], fraction=0.046, pad=0.04)

for j in range(3):
    axes[0, j].set_title(titles_col[j], fontsize=13, fontweight='bold')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
comp3way_path = os.path.join(RESULTS_DIR, 'SEN2SR_Full_10Band_3Way_Comparison.png')
plt.savefig(comp3way_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'3-way comparison figure saved to: {comp3way_path}')


## Cell 9 — Spatial Edge Acuity & Gradient Sharpness Evaluation


In [ ]:
import pandas as pd
from scipy.ndimage import laplace, sobel

def compute_spatial_sharpness(img_tensor):
    r, g, b = img_tensor[BAND_MAP['B04']].cpu().numpy(), img_tensor[BAND_MAP['B03']].cpu().numpy(), img_tensor[BAND_MAP['B02']].cpu().numpy()
    gray = 0.299 * r + 0.587 * g + 0.114 * b

    lap_var = np.var(laplace(gray))

    sx = sobel(gray, axis=0)
    sy = sobel(gray, axis=1)
    sobel_mean = np.mean(np.sqrt(sx**2 + sy**2))

    return lap_var, sobel_mean

lap_nat, sob_nat = compute_spatial_sharpness(X)
lap_base, sob_base = compute_spatial_sharpness(output_baseline)
lap_fine, sob_fine = compute_spatial_sharpness(output_finetuned)

boost_lap = ((lap_fine - lap_base) / (lap_base + 1e-8)) * 100
boost_sob = ((sob_fine - sob_base) / (sob_base + 1e-8)) * 100

metrics_df = pd.DataFrame([
    {'Resolution / Model': 'Native Sentinel-2 (10m)', 'Laplacian Variance (Acuity)': f'{lap_nat:.6f}', 'Sobel Gradient Energy': f'{sob_nat:.6f}', 'Relative Acuity Gain': '0.0% (Reference)'},
    {'Resolution / Model': 'Pretrained Baseline Full (2.5m)', 'Laplacian Variance (Acuity)': f'{lap_base:.6f}', 'Sobel Gradient Energy': f'{sob_base:.6f}', 'Relative Acuity Gain': 'Baseline'},
    {'Resolution / Model': 'Fine-Tuned Full SEN2SR (2.5m)', 'Laplacian Variance (Acuity)': f'{lap_fine:.6f}', 'Sobel Gradient Energy': f'{sob_fine:.6f}', 'Relative Acuity Gain': f'+{boost_lap:.2f}% over Baseline'}
])

print('\n=== Quantitative Edge Acuity Benchmark ===')
print(metrics_df.to_string(index=False))

csv_path = os.path.join(RESULTS_DIR, 'SEN2SR_Full_Edge_Acuity_Improvement.csv')
metrics_df.to_csv(csv_path, index=False)
print(f'Metrics CSV saved to: {csv_path}')


## Cell 10 — Spectral Difference Maps & Index Residual Distributions


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

diff_ndvi = (layers_finetuned['ndvi'] - layers_baseline['ndvi']).cpu().numpy()
diff_ndwi = (layers_finetuned['ndwi'] - layers_baseline['ndwi']).cpu().numpy()
diff_ndbi = (layers_finetuned['ndbi'] - layers_baseline['ndbi']).cpu().numpy()

diffs = [
    ('NDVI Difference (FineTuned - Baseline)', diff_ndvi, 'RdBu', (-0.3, 0.3)),
    ('NDWI Difference (FineTuned - Baseline)', diff_ndwi, 'RdBu', (-0.3, 0.3)),
    ('NDBI Difference (FineTuned - Baseline)', diff_ndbi, 'RdBu', (-0.3, 0.3))
]

for i, (title, arr, cmap, lims) in enumerate(diffs):
    im = axes[i, 0].imshow(arr, cmap=cmap, vmin=lims[0], vmax=lims[1])
    axes[i, 0].set_title(title, fontsize=12, fontweight='bold')
    axes[i, 0].axis('off')
    fig.colorbar(im, ax=axes[i, 0], fraction=0.046, pad=0.04)

    vals = arr[np.isfinite(arr)]
    axes[i, 1].hist(vals, bins=60, color='royalblue', alpha=0.7, edgecolor='black')
    axes[i, 1].set_title(f'{title} Distribution', fontsize=12, fontweight='bold')
    axes[i, 1].set_xlabel('Index Delta'); axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
diff_plot_path = os.path.join(RESULTS_DIR, 'SEN2SR_Full_10Band_Difference_Maps.png')
plt.savefig(diff_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Difference maps saved to: {diff_plot_path}')


## Cell 11 — Export High-Resolution 10-Band GeoTIFF to Google Drive


In [ ]:
EXPORT_TIF = os.path.join(RESULTS_DIR, 'sen2sr_full_finetuned_10band.tif')

out_np = output_finetuned.cpu().numpy()
export_profile = ref_profile.copy()
scale = 4

export_profile.update(
    driver='GTiff',
    count=10,
    height=out_np.shape[1],
    width=out_np.shape[2],
    dtype='float32',
    nodata=None,
    compress='deflate',
    transform=ref_transform * Affine.scale(1/scale, 1/scale)
)

with rasterio.open(EXPORT_TIF, 'w', **export_profile) as dst:
    dst.write(out_np.astype(np.float32))

print(f'Successfully exported 10-Band 2.5m Super-Resolved GeoTIFF to Google Drive:')
print(f'  Path: {EXPORT_TIF}')
print(f'  Bands: 10 | Shape: ({out_np.shape[1]}, {out_np.shape[2]}) | Scale: 2.5m GSD')
